In [0]:
import base64
import gzip
import hashlib
import io
import tarfile

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("silver_update_id", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
assert RUN.startswith("dq4_silver_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SILVER_UPDATE_ID
LANE = "dq4_profile"
PAYLOAD = """H4sIAAAAAAAAA+1a23LbNhD1s74CbyQ7qkrRkhynjmd8YRzNuHIqK5PpE0OTsMKWBGQCVO1OP74LkJJI6lbHtjzJ7HngBcQCi8Xu4gCSuIt/EdKXwgviiEWBHy8eRMBT2hJ38d7TYAN6vY66tw+6dvmu0HHgud11OvvOgePYvT34erC/v0fsJ/b7v5DB6FNC9qgvObtZX2/b9+8UZ0P3ZOSSqyEZuh8vT85cMjo5vXTJlzdeSKdfWiKKpzT17gJPJpNWeNfxNnqLp2vkMo7t9Ow3TpecXDc+90cfyI0vKLwQs0HItXvpno1IwP2YioCaAWeBL02DJ3zy1miSMZXmbRRLmpq3KU+8PwVnpuT5Pe8q4CG1msTw09R/OBIyzQJ5BIURG+tvBIrguUmKMvEgJE3mpZHwBM/SgJIbzmPqs+NjA5q7Jz8fk/tWVeYdMbKUvdXKKU3pRHpRqKrbVqvUpaU6ywci0kCNYz7AvC+v0i4o73mMM+p5qi3j39UCtFLPspQNZ1r8RR+aYM4CdeEwEpPYf6jIl6Vn31e0wLI4jm7NmP8NUwAmS8yMRbIwmTf144xaSmeteLV1VbHUpJZbWG2DureUhkutlb8tRM9Orkdm6EvqwbyzwDQSzuRXsB+dUgYDgy8ySqhu4Rw8XD/oOvVG9FA8liU3NNW1rz5BAOj6U6j6fnj1G+l4k5SHRTC01vh+w2rMvXoxOSTx782avVfOQdVsSqpmuLltVxkz4BmT5nn/etQfgAarRJlXLhVzUbCCSz5/cAdknPJsAu5Zng8LnN8mI/XZUO/GwnyAqpi2b1kgn5SKhHsJvRmFEgZxB+dKtximLX68RqXSWUur3GVFe3VV9fusjaqf5Kb9KbdhrXRaWBbch6ZRMP/qT8Bh7r0JBV2YjGJqTptEJyrTbqn1DzJHa3bTV6errvpyoK+Hh/l1djvMA/9u0cd0XPSfQPZqQmILIWUXRfnLvK7IEnNhhSmM+zbmPNWVYcz+jVCPR+SQtru5Qdr5TNnKJsUgIya/qcHK/GuoKpOEh6aOwKIy9HHav+gPRpBR2nZpblaoIkN7t6qst4oMu2tVOdowBEbHa+X6A2LqmT+cXw+t9S1FjIoN5livwz805Q2d4dTa3LgYXn36SE7/IPqhP7iAZXoE67VZSWgqi1jNZUsWWK7cLAflIyXzOLV+fW2e9KNCLPP/iS+/8piPH7yUiiyWT94BbOP/+067yv8de7/XRf6/CzyZ/9e9ZTc7gKIv3ALseAsQZEk+yhn1x23C9m1CPURwo4AbBdwo4EYBNwq4UXhtrOD/00iqI61ozJ7h7F9h6/l/Z+n8v+3YyP93gSfz/4W37Ib55/0h8cez/9cl9Qu/RzqPdB7pPNJ5pPNI5787KP4PKQvoFoU4FzSVz/9HoC383+51O7Xz/3bPwfP/naA/uHaHI0g8o6tlzg98P2d7XuEiM6JjrKD6wL+MNa5jNKvUaDMFqlOdJQJTYw0VApkv4bBelxZnWDsrS+VcMl/aimWlSOlFhp6n1yJbzmWM8987ajxZmiqyqYgmeFAyMa08oT7jH6cakMmHLmHk+B2sRvaL5MJN8f9cPwRujf9ebf/vOE4X438neKH4r7sOZoBv/On0pXPApvh/roPArfHfWVr/7baD8b8LvFD8L1wHI//Rh6a7WPcRCAQCgUAgEAgEAoFAIBAIBAKBQCAQCAQCgUD8OPgP1l0TfgBQAAA="""

def qs(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

def adapt(value):
    return (value
        .replace("dq4_silver_20260825", RUN)
        .replace("2026-08-25T10:50:49.897Z", RUN_OPEN_TS)
        .replace("f5c7c7ab-e37d-4a31-b9c2-b7631becb16a", SILVER_UPDATE_ID))

def unpack():
    archive = tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz")
    items = []
    for member in archive.getmembers():
        if member.isfile() and member.name.endswith(".sql"):
            items.append((member.name, adapt(archive.extractfile(member).read().decode("utf-8"))))
    return sorted(items, key=lambda x: (0 if "/stats_" in x[0] else 1, x[0]))

def execute(seq, name, sql):
    sha = hashlib.sha256(sql.encode()).hexdigest()
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(sql).collect()
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'ok',NULL,NULL,current_timestamp(),current_timestamp(),'DQ4')""")
    except Exception as exc:
        msg = str(exc)[:4000]
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'error',{qs(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
        raise

run_row = spark.sql(f"SELECT count(*) n FROM 8_dev.silver_qc.dq_run WHERE run_id={qs(RUN)} AND finished_at IS NULL").first().n
assert run_row == 1, "run_id must identify one open dq_run row"

existing = spark.sql(f"SELECT count(*) n FROM 8_dev.silver_qc.dq_value_profile WHERE run_id={qs(RUN)}").first().n
assert existing == 0, f"dq_value_profile already contains {existing} rows for {RUN}"

items = unpack()
for seq, (name, sql) in enumerate(items):
    execute(seq, name, sql)

print({"run_id": RUN, "lane": LANE, "statements": len(items), "status": "ok"})